# Ingest to Bronze Job

1. **Read Landing XLSX Sheets to DataFrames:**
   - Informática e comunicação
   - Governança
   - Dicionário

   **Note:** The Dicionário worksheet contains variable definitions (business glossary) for coded column names (e.g., MTIC001) used across the other sheets.

2. **Clean & Standardize Dictionary:**
   - Handle empty cell bindings and structural padding resulting from the Excel source.

3. **Persist Tables to Bronze Schema using Spark:**
   - munic_raw_data
   - munic_raw_dictionary

In [0]:
!pip install openpyxl

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart Python Kernel to use the openpyxl
dbutils.library.restartPython()

In [0]:
%run ../environment/env_variables

In [0]:
%run ../environment/env_functions

In [0]:
import pandas as pd
import openpyxl
import requests
import re #regex

## Envirnoment Variables

- This notebook is made to be imported by the jobs of the Munic Pipeline
- The objective is set common variables used between the pipeline to easy the management of them
and centralize the definition in one place, avoiding typo errors along the pipeline

# env_functions

- This notebook is to sabe methods used along all the Munic Pipeline

In [0]:
# Metadata Variables
dataset_name = "IBGE_MUNIC"
reference_year = "2024"


In [0]:
# Paths of where the final tables are going to be stored 
# Colocar o nome do ano no nome da tabela (verificar se faremos isso)
bronze_tbl_raw_dict = f'{catalog_name}.{bronze_schema}.munic_raw_dict'
bronze_tbl_raw_info = f'{catalog_name}.{bronze_schema}.munic_raw_info_data'
bronze_tbl_raw_gov = f'{catalog_name}.{bronze_schema}.munic_raw_gov_data'


## 1. **Read Landing XLSX Sheets to DataFrames:**

Workbook (wb) = the complete file

Worksheet (ws) = the individual tabbed pages inside the workbook

In [0]:
# Transform each sheet to pandas.DataFrame

# Dictionary of the variables into the data sheets 
df_dict = pd.read_excel(munic_file, sheet_name="Dicionário", header=None)

# Cities Computer data
df_info_ws = pd.read_excel(
    munic_file,
    sheet_name="Informática e comunicação"
)

# Cities Governance data 
df_gov_ws = pd.read_excel(
    munic_file,
    sheet_name="Governanca"
)


In [0]:
df_dict.head()

,0,1,2,3,4,5
0,NaN,NaN,NaN,NaN,NaN,NaN
1,Temas disponíveis,NaN,NaN,NaN,NaN,variável
2,NaN,NaN,NaN,NaN,NaN,NaN
3,1. RECURSOS HUMANOS POR VÍNCULO,NaN,NaN,NaN,NaN,título link
4,2. INFORMÁTICA E COMUNICAÇÃO,NaN,NaN,NaN,NaN,título link


#### 2. **Clean & Standardize Dictionary Dataframe:**

In [0]:
def obtain_filled_cells(row):
    # Basically catch the values in a row from the dictionary sheet, to retire the columns vacances
    filled_cells = [str(val).strip() for val in row if pd.notna(val) and str(val).strip() != '']

    return filled_cells

In [0]:
def extract_var_code_meaning(row, var_pattern_list):
  """
    Extracts variable codes and their corresponding survey descriptions
    from a raw IBGE MUNIC dictionary row.

    Parameters:
        row (pd.Series): A single row from the unformatted dictionary sheet.
        var_pattern (str): The thematic variable prefix to match (e.g., 'MTIC', 'MGOV', 'MREH').

    Returns:
        dict: {'cod_variable': <str>, 'meaning': <str>} if a code matching the pattern is found,
              otherwise None.
    """

  # It´s used to save the pattern matched on the row using regex
  code_matches = list()
  filled_cells = obtain_filled_cells(row)

  # Verify if the row has values or not
  if not filled_cells:
      return None

  # Create the Regex Expressions
  regex_expr_list = [rf'^{pattern}\d+' for pattern in var_pattern_list]


  # Match if any cell have some pattern codes and add it in code_matches list
  for re_expr in regex_expr_list:
      for val in filled_cells:
        if re.match(re_expr, val, flags=re.IGNORECASE):
           code_matches.append(val)


  # If has any value with some of the code patterns
  if code_matches:
      cod_var = code_matches[0]

      # Find the description or what means of the MTIC code
      desc_cells = [val for val in filled_cells if val != cod_var]

      # If has one string value, get it. If has more, join the them. If doesn´t have, return None meaning to that code
      if desc_cells:
        if len(desc_cells) == 1:
          code_meaning = desc_cells[0]
        else:
          code_meaning = " - ".join(desc_cells) if desc_cells else None

      return {"cod_variable": cod_var, "meaning": code_meaning}

  # Else, returns None
  return None

In [0]:
code_patterns = ['MTIC', 'MGOV']

# Get the meaning of the codes in the "Dicionario" worksheet (Ws)
# The result is a list of dicts (each dict is a code and its meaning)
meanings_raw_dict = [res for res in df_dict.apply(extract_var_code_meaning, axis=1, args=(code_patterns,)) if res is not None]

In [0]:
# Transform on Dataframe
df_raw_dict = pd.DataFrame(meanings_raw_dict)

In [0]:
df_raw_dict.head()

,cod_variable,meaning
0,MTIC011,Correio
1,MTIC012,Jornais
2,MTIC013,Websites
3,MTIC014,WhatsApp
4,MTIC015,Telefone


## 3. **Persist Tables to Bronze Schema using Spark:**

In [0]:
df_gov_ws.dtypes

Cod Munic     int64
Uf           object
Cod Uf        int64
Desc Mun     object
Populacao     int64
              ...  
Mgov32       object
Mgov33       object
Mgov34       object
Mgov35       object
Mgov36       object
Length: 169, dtype: object

In [0]:
df_info_ws.dtypes

Cod Munic     int64
Uf           object
Cod Uf        int64
Desc Mun     object
Populacao     int64
              ...  
Mtic383      object
Mtic384      object
Mtic385      object
Mtic386      object
Mtic387      object
Length: 205, dtype: object

In [0]:
def prepare_data_to_delta_format(df):
    """
    Prepare a pandas DataFrame for ingestion into Delta Lake:
    - Standardizing column names.
    - Converting all object type columns to pandas 'string' dtype.

    Parameters:
        df (pd.DataFrame): The input pandas DataFrame.

    Returns:
        pd.DataFrame: The prepared pandas DataFrame
    """

    # Standardize Pandas column names 
    df.columns = [
        col.strip()
        .replace(" ", "_")
        .replace(";", "_")
        .replace(",", "_")
        .replace("-", "_")
        .lower() 
        for col in df.columns
    ]

    # Convert all object type columns to pandas 'string' dtype
    # It´s done to Spark don´t throw a Object inference error at the pass of pandas dataframe to spark dataframe 
    for col in df.select_dtypes(include='object').columns:
         df[col] = df[col].astype('string')

In [0]:
prepare_data_to_delta_format(df_gov_ws)
prepare_data_to_delta_format(df_info_ws)

In [0]:
df_gov_ws.head()

,cod_munic,uf,cod_uf,desc_mun,populacao,faixa_populacao,regiao,mgov01,mgov011a,mgov011b,mgov0121,mgov0122,mgov021,mgov022,mgov023,mgov024,mgov025,mgov026,mgov027,mgov028,mgov031,mgov032,mgov033,mgov034,mgov035,mgov036,mgov04,mgov05,mgov06,mgov061,mgov0621,mgov0622,mgov0623,mgov0711,mgov0712,mgov0713,mgov0715,mgov0721,mgov0722,mgov0723,...,mgov243,mgov244,mgov245,mgov246,mgov247,mgov248,mgov251,mgov252,mgov253,mgov254,mgov255,mgov26,mgov261a,mgov261b,mgov261c,mgov262,mgov271,mgov272,mgov273,mgov274,mgov28,mgov29,mgov301,mgov302,mgov303,mgov31,mgov3111,mgov3112,mgov3113,mgov3114,mgov3115,mgov3116,mgov3117,mgov3118,mgov3119,mgov32,mgov33,mgov34,mgov35,mgov36
0,1100015,RO,11,Alta Floresta DOeste,22853,4 - 20001 até 50000,1 - Norte,Não,-,-,-,-,-,-,-,-,-,-,-,-,Não,Sim,Não,Não,Sim,Não,Sim,Não publica,Sim,Não,Não,Sim,Não,Não,Sim,Não,Não,Não,Sim,Não,...,Não,Não,Não,Não,Não,Sim,Não,Não,Não,Não,Sim,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Não,-,Não,Não,Não
1,1100023,RO,11,Ariquemes,108573,6 - 100001 até 500000,1 - Norte,Sim,13412,2017,Não,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Sim,Sim,Sim,Sim,Não,Não,Não,Sim,Não publica,Sim,Sim,Não,Sim,Não,Sim,Sim,Não,Não,Sim,Sim,Não,...,Não,Sim,Sim,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Sim,Na Controladoria ou Corregedoria municipal,Sim,Sim,Sim
2,1100031,RO,11,Cabixi,5690,2 - 5001 até 10000,1 - Norte,Sim,148,2016,Não,Sim,Sim,Sim,Não,Sim,Não,Sim,Sim,Sim,Sim,Sim,Sim,Não,Não,Não,Não,Não publica,Sim,Sim,Não,Sim,Não,Não,Sim,Não,Não,Sim,Não,Não,...,Não,Sim,Não,Sim,Não,Sim,Não,Não,Não,Não,Sim,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Sim,No gabinete do(a) Prefeito(a),Não,Sim,Sim
3,1100049,RO,11,Cacoal,97637,5 - 50001 até 100000,1 - Norte,Sim,6544,2017,Não,Sim,Sim,Sim,Sim,Sim,Não,Sim,Sim,Sim,Sim,Não,Não,Não,Não,Não,Sim,Perguntas e respostas,Sim,Sim,Não,Sim,Não,Sim,Sim,Não,Não,Sim,Sim,Não,...,Não,Sim,Sim,Não,Sim,Sim,Não,Não,Não,Não,Sim,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Sim,No departamento ou área de Tecnologia da Infor...,Não sabe informar,Não,Não
4,1100056,RO,11,Cerejeiras,16975,3 - 10001 até 20000,1 - Norte,Sim,152,2016,Não,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Sim,Sim,Sim,Não,Não,Não,Não,Sim,Perguntas e respostas,Sim,Sim,Não,Sim,Não,Sim,Sim,Não,Não,Sim,Sim,Não,...,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Sim,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Não,-,Não,Não,Não


### Spark Processment and write of the bronze tables

In [0]:
spark_df_gov = spark.createDataFrame(df_gov_ws)
spark_df_info = spark.createDataFrame(df_info_ws)

In [0]:
spark_df_raw_dict = spark.createDataFrame(df_raw_dict)

#### Add metadata columns

In [0]:
# Help to add the ingestion_timestamp and the source_file
from pyspark.sql import functions as F

def add_ingestion_metadata(df, dataset_name, source_file, reference_year, source_url):
    return (
        df.withColumn("source_file", F.lit(source_file))
          .withColumn("source_url", F.lit(source_url))
          .withColumn("dataset_name", F.lit(dataset_name))
          .withColumn("reference_year", F.lit(reference_year))
          .withColumn("ingestion_timestamp", F.current_timestamp())
    )

In [0]:
# Variables defines at the beginning of the dataset
spark_df_raw_info_governed = add_ingestion_metadata(
        spark_df_info, 
        dataset_name=dataset_name, 
        source_file=munic_file, 
        reference_year=reference_year,
        source_url=munic_source_url
    )

spark_df_raw_gov_governed = add_ingestion_metadata(
        spark_df_gov, 
        dataset_name=dataset_name, 
        source_file=munic_file, 
        reference_year=reference_year,
        source_url=munic_source_url
    )

In [0]:
spark_df_raw_dict_governed = add_ingestion_metadata(
        spark_df_raw_dict, 
        dataset_name=dataset_name, 
        source_file=munic_file, 
        reference_year=reference_year,
        source_url=munic_source_url
    )

In [0]:
display(
    spark_df_raw_gov_governed
        .select("source_file",
                 "ingestion_timestamp",
                 "dataset_name",
                 "reference_year",
                 "source_file")
        .show(5)
)

+--------------------+--------------------+------------+--------------+
|         source_file| ingestion_timestamp|dataset_name|reference_year|
+--------------------+--------------------+------------+--------------+
|/Volumes/munic_ca...|2026-10-07 17:45:...|  IBGE_MUNIC|          2024|
|/Volumes/munic_ca...|2026-10-07 17:45:...|  IBGE_MUNIC|          2024|
|/Volumes/munic_ca...|2026-10-07 17:45:...|  IBGE_MUNIC|          2024|
|/Volumes/munic_ca...|2026-10-07 17:45:...|  IBGE_MUNIC|          2024|
|/Volumes/munic_ca...|2026-10-07 17:45:...|  IBGE_MUNIC|          2024|
+--------------------+--------------------+------------+--------------+
only showing top 5 rows


#### Save the tables on bronze schema

In [0]:
# --- Save to bronze using MERGE (upsert), partitioned by ano_edicao ---
# MERGE avoids duplicates: matched rows are updated, new rows are inserted.
# Manual schema evolution (ALTER TABLE) lets columns from different survey editions
# (e.g. MTIC015 in 2024 vs MTIC016 in 2025) coexist in the same table.
# upsert_delta_table is imported from env_functions notebook

ano_edicao = int(reference_year) if isinstance(reference_year, (int, float, str)) else 2024

# Data tables — merge key: ano_edicao + cod_munic (municipality PK)
upsert_delta_table(
    spark_df_raw_info_governed,
    bronze_tbl_raw_info,
    ["ano_edicao", "cod_munic"],
    ano_edicao,
    f"Ingested raw MUNIC {ano_edicao} survey data — Informática e Comunicação sheet"
)

upsert_delta_table(
    spark_df_raw_gov_governed,
    bronze_tbl_raw_gov,
    ["ano_edicao", "cod_munic"],
    ano_edicao,
    f"Ingested raw MUNIC {ano_edicao} survey data — Governança sheet"
)

MERGE done: munic_catalog.bronze.munic_raw_info_data - year 2024
MERGE done: munic_catalog.bronze.munic_raw_gov_data - year 2024


In [0]:
# Dictionary table — merge key: ano_edicao + cod_variable (variable PK)
# Each edition gets its own versioned dictionary, so codes with different meanings
# across years coexist safely.
upsert_delta_table(
    spark_df_raw_dict_governed,
    bronze_tbl_raw_dict,
    ["ano_edicao", "cod_variable"],
    ano_edicao,
    f"Ingested dictionary of variables — MUNIC {ano_edicao} survey"
)

MERGE done: munic_catalog.bronze.munic_raw_dict - year 2024
